# Etive training metrics

Interactive diagnostics for one or more training runs. Comment or uncomment entries in `RUNS` below to choose which runs to compare.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook", palette="colorblind")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (11, 5)

In [ ]:
repo_root = Path.cwd()
if not (repo_root / "Cargo.toml").exists():
    repo_root = repo_root.parent

# Comment out a line to hide that run; uncomment it to include it in every view.
RUNS = {
    "4x64 corrected": repo_root / "benchmarks/training-regression/corrected-4x64-256-4h.csv",
    "8x64 LOS": repo_root / "benchmarks/training-regression/trial-8x64-256-8h.csv",
    # "another run": repo_root / "benchmarks/another-metrics.csv",
}

missing = [path for path in RUNS.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing metrics files: {missing}")
if not RUNS:
    raise ValueError("Uncomment at least one entry in RUNS")

frames = []
for run, path in RUNS.items():
    frame = pd.read_csv(path)
    frame["run"] = run
    frames.append(frame)
    print(f"Loaded {len(frame):,} generations for {run} from {path}")
metrics = pd.concat(frames, ignore_index=True, sort=False)
metrics["training_steps_per_second"] = (
    metrics["training_steps"] / metrics["training_seconds"]
)
metrics.groupby("run", sort=False).tail(3)

## Training and validation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

policy = metrics.melt(
    id_vars=["generation", "run"],
    value_vars=["policy_kl", "validation_policy_kl"],
    var_name="split",
    value_name="loss",
)
policy["split"] = policy["split"].map({"policy_kl": "Training", "validation_policy_kl": "Validation"})
sns.lineplot(data=policy, x="generation", y="loss", hue="run", style="split", ax=axes[0])
axes[0].set(title="Policy KL", xlabel="Generation", ylabel="KL divergence")

value = metrics.melt(
    id_vars=["generation", "run"],
    value_vars=["value_loss", "validation_value_loss"],
    var_name="split",
    value_name="loss",
)
value["split"] = value["split"].map({"value_loss": "Training", "validation_value_loss": "Validation"})
sns.lineplot(data=value, x="generation", y="loss", hue="run", style="split", ax=axes[1])
axes[1].axhline(1.0, color="0.35", linestyle="--", linewidth=1, label="Constant prediction")
axes[1].set(title="Value loss", xlabel="Generation", ylabel="MSE")
axes[1].legend()
sns.despine(fig)
fig.suptitle("Training run comparison", fontweight="bold")
fig.tight_layout()

## Arena promotions

In [ ]:
evaluations = metrics.loc[metrics["evaluated"], [
    "run", "generation", "score", "current_wins", "previous_wins", "draws", "promoted"
]].copy()

fig, ax = plt.subplots(figsize=(11, 5))
sns.lineplot(data=evaluations, x="generation", y="score", hue="run", marker="o", ax=ax)
ax.axhline(0.5, color="0.35", linestyle="--", linewidth=1, label="Even score")
ax.set(title="Candidate score against current champion", xlabel="Generation", ylabel="Score", ylim=(0, 1.05))
ax.legend()
sns.despine(fig)
fig.tight_layout()
evaluations

## Throughput and replay

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
sns.lineplot(data=metrics, x="generation", y="self_play_evaluations_per_second", hue="run", ax=axes[0])
axes[0].set(title="Self-play throughput", xlabel="Generation", ylabel="Evaluations/s")
sns.lineplot(data=metrics, x="generation", y="training_steps_per_second", hue="run", ax=axes[1], legend=False)
axes[1].set(title="Training throughput", xlabel="Generation", ylabel="Steps/s")
sns.lineplot(data=metrics, x="generation", y="replay_samples", hue="run", ax=axes[2], legend=False)
axes[2].set(title="Replay occupancy", xlabel="Generation", ylabel="Positions")
sns.despine(fig)
fig.tight_layout()

## Latest state

In [ ]:
latest = metrics.groupby("run", sort=False).tail(1).set_index("run")
latest[[
    "generation",
    "champion_generation",
    "replay_samples",
    "policy_kl",
    "validation_policy_kl",
    "value_loss",
    "validation_value_loss",
    "self_play_evaluations_per_second",
]]